* Compare the performance of different portfolio optimizers on problems of different sizes.

* The results compare `alphamind` with other Python optimizers. The current implementation uses `cvxpy`; the historical comparison preferred `cvxopt` and then `scipy`.

* `scipy` performs poorly on `ashare_ex`, so its results for that universe are generally omitted.

* All times are in milliseconds.

* The original example asked users to set `DB_URI` to a database. This version uses deterministic local data, so neither `DB_URI` nor a database is required.

In [ ]:
import time


## 0. Data Preparation
------------------

In [ ]:
import numpy as np
import pyarrow as pa


## 1. Linear Optimization (Linear Constraints)
---------------------------------

In [ ]:
from alphamind.portfolio.linearbuilder import linear_builder


In [ ]:
from alphamind.portfolio.meanvariancebuilder import mean_variance_builder

rng = np.random.default_rng(42)


In [ ]:
n = 30


## 2. Linear Optimization (L1 Constraints)
-----------------------

In [ ]:
er = rng.normal(0.0, 0.01, size=n)


In [ ]:
benchmark = np.full(n, 1.0 / n)
risk = np.ones((n, 1))


## 3. Mean-Variance Optimization (Unconstrained)
-----------------------

In [ ]:
target = (np.array([1.0]), np.array([1.0]))


In [ ]:
covariance = np.eye(n) * 0.02


## 4. Mean-Variance Optimization (Box Constraints)
---------------

In [ ]:
risk_model = {"cov": covariance, "factor_cov": None, "factor_loading": None, "idsync": None}
started = time.perf_counter()


In [ ]:
linear_status, _, linear_weights = linear_builder(er, 0.0, 1.0, risk, target)


## 5. Mean-Variance Optimization (Box and Linear Constraints)
----------------

In [ ]:
linear_seconds = time.perf_counter() - started


In [ ]:
started = time.perf_counter()
mv_status, _, mv_weights = mean_variance_builder(
    er, risk_model, benchmark, 0.0, 1.0, risk, target
)


## 6. Linear Optimization (Quadratic Constraints)
-------------------------

In [ ]:
mv_seconds = time.perf_counter() - started


In [ ]:
assert linear_status in ("optimal", "optimal_inaccurate")
assert mv_status in ("optimal", "optimal_inaccurate")


In [ ]:
pa.table({"method": ["linear", "mean_variance"], "seconds": [linear_seconds, mv_seconds]})
